In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning4/sample_submission.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning4/train.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning4/test.csv
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__results__.html
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__notebook__.ipynb
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__output__.json
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/split.csv
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/custom.css
/kaggle/input/notebooks/lyk1zm/03-transformer-training/transformer_validation_predictions.csv
/kaggle/input/notebooks/lyk1zm/03-transformer-training/preference_data.py
/kaggle/input/notebooks/lyk1zm/03-transformer-trainin

In [2]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from tqdm.auto import tqdm


device = torch.device("cuda")

config_paths = list(
    Path("/kaggle/input").rglob("inference_config.json")
)

assert len(config_paths) == 1, (
    "Нужен ровно один подключённый артефакт нашей модели. "
    f"Найдено: {config_paths}"
)

MODEL_DIR = config_paths[0].parent

with open(
    MODEL_DIR / "inference_config.json",
    encoding="utf-8",
) as file:
    config = json.load(file)

assert not config["smoke_test"], (
    "Подключена модель короткого smoke test, а не полного обучения."
)

shutil.copy2(
    MODEL_DIR / "preference_data.py",
    "/kaggle/working/preference_data.py",
)

sys.path.insert(0, "/kaggle/working")

from preference_data import (
    prepare_frame,
    encode_frame,
    PreferenceDataset,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
).to(device)

model.eval()

print("Loaded:", MODEL_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Swap TTA:", config["use_swap_tta"])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded: /kaggle/input/notebooks/lyk1zm/03-transformer-training/preference_encoder
GPU: Tesla T4
Swap TTA: True


In [3]:
possible_dirs = [
    Path("/kaggle/input/competitions/llm-classification-finetuning"),
    Path("/kaggle/input/llm-classification-finetuning"),
]

DATA_DIR = next(
    (p for p in possible_dirs if (p / "test.csv").exists()),
    None,
)

assert DATA_DIR is not None

test = pd.read_csv(DATA_DIR / "test.csv")
test = prepare_frame(test)

encoded = encode_frame(
    test,
    tokenizer,
    prompt_tokens=config["prompt_tokens"],
    answer_tokens=config["answer_tokens"],
)

dataset_kwargs = {
    "cls_token_id": tokenizer.cls_token_id,
    "sep_token_id": tokenizer.sep_token_id,
}

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
)


def make_loader(fixed_swap=False):
    dataset = PreferenceDataset(
        encoded,
        fixed_swap=fixed_swap,
        **dataset_kwargs,
    )

    return DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )


print("Test rows:", len(test))

Test rows: 3


In [4]:
@torch.inference_mode()
def predict(loader):
    parts = []

    for batch in tqdm(loader):
        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(**batch).logits

        parts.append(
            torch.softmax(logits.float(), dim=-1)
            .cpu()
            .numpy()
        )

    return np.vstack(parts)


probabilities = predict(
    make_loader(fixed_swap=False)
)

if config["use_swap_tta"]:
    swapped_probabilities = predict(
        make_loader(fixed_swap=True)
    )[:, [1, 0, 2]]

    probabilities = (
        probabilities + swapped_probabilities
    ) / 2

probabilities = np.asarray(
    probabilities,
    dtype=np.float64,
)

probabilities = np.clip(
    probabilities,
    1e-15,
    1.0,
)

probabilities /= probabilities.sum(
    axis=1,
    keepdims=True,
)

target_columns = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

submission = pd.DataFrame(
    probabilities,
    columns=target_columns,
)

submission.insert(
    0,
    "id",
    test["id"].to_numpy(),
)

assert len(submission) == len(test)
assert submission["id"].is_unique
assert submission["id"].tolist() == test["id"].tolist()
assert np.isfinite(probabilities).all()
assert (probabilities >= 0).all()
assert np.allclose(probabilities.sum(axis=1), 1.0)

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False,
)

print(submission.head())
print("Saved /kaggle/working/submission.csv")

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.197760        0.221455    0.580784
1   211333        0.330195        0.359769    0.310036
2  1233961        0.347459        0.359199    0.293343
Saved /kaggle/working/submission.csv
